In [1]:

# Import required packages
import os
from nltk.corpus import stopwords
import nltk
import string
from collections import defaultdict
import pandas as pd
from gensim.models.phrases import Phrases, Phraser
from gensim.utils import simple_preprocess
import spacy
import operator
import plotly as py
import plotly.express as px
import re
import math

In [2]:
# only need to do this once
nltk.download('stopwords') 
import spacy.cli
spacy.cli.download('en_core_web_sm')

[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 71.6 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
# Configure data and results directories
# TODO Naming
homePath = os.path.abspath(os.getcwd())
data_path = os.path.join(homePath, 'data') # TODO Inputs
resultsPath = os.path.join(homePath, 'results') # TODO Inputs
os.makedirs(resultsPath, exist_ok=True)


In [4]:
# Set variables for processing
# TODO Naming
stop_language = 'english'
lemLang_model = "en_core_web_sm"
encoding = "utf-8"
how_to_handleErrors = "ignore"

In [5]:
# Create the list of stop words, or words that are not important for our analysis and should be excluded
stopWords = []
stopWords.extend(stopwords.words(stop_language)) # Add nltk stop words
# Add custom stop words
stopWords.extend(['would', 'said', 'says'])
stopWords.extend(['also', 'good', 'lord'])
stopWords.extend(['come', 'let','say', 'speak', 'know']) 
stopWords.extend(['hamlet']) # TODO DRY
stopWordsFilepath = os.path.join(data_path, "earlyModernStopword.txt")
with open(stopWordsFilepath, "r",encoding = encoding) as stopfile:
    stopWordsCustom = []
    for x in stopfile.readlines():
        stopWordsCustom.append(x.strip())

stopWords.extend(stopWordsCustom)

In [6]:
# Variables for making Trigrams
minCount = 5
thresHold = 100

def makeTrigrams(tokens):
    """Build bigram and trigram models from tokenized text.

    Parameters
    ----------
    tokens : list of list of str
        Tokenized documents, each document is a list of word tokens.

    Returns
    -------
    list of list of str
        Documents with words grouped into bigrams and trigrams.
    """
    # Build the bigram and trigram models
    # get bigram phrases
    bigram = Phrases(tokens, min_count=minCount, threshold=thresHold) # higher threshold indicates fewer phrases  
    bigram_model = Phraser(bigram) # Removes model state from Phrases to reduce memory use
    # get trigram phrases
    bigram = Phrases(tokens, min_count=minCount, threshold=thresHold) # TODO DRY
    trigram = Phrases(bigram[tokens], threshold=thresHold)
    trigram_model = Phraser(trigram)
    result = []
    for doc in tokens:
        bigram_result = bigram_model[doc]
        trigram_result = trigram_model[bigram_result]
        result.append(trigram_result)
    return result

In [7]:
# Lemmatize tokenized text using spaCy
def makeLemma(tokens):
    """Lemmatize tokenized text using spaCy after building n-grams.

    Parameters
    ----------
    tokens : list of list of str
        Tokenized documents to be lemmatized.

    Returns
    -------
    list of list of str
        Lemmatized tokens with n-gram groupings preserved.
    """
    dataWordsNgrams = makeTrigrams(tokens)

    def lemmatization(tokens):
        """Lemmatize a list of tokenized sentences using spaCy.

    Parameters
    ----------
    tokens : list of list of str
        Sentences with each sentence as a list of word tokens.

    Returns
    -------
    list of list of str
        Lemmatized tokens, excluding pronoun lemmas.
    """
        textsOut = []
        for sent in tokens:
            doc = nlp(" ".join(sent)) 
            textsOut.append([token.lemma_ for token in doc if token.lemma_ != '-PRON-'])
        return textsOut
    # Initialize spacy language model, removing the parser and ner components
    nlp = spacy.load(lemLang_model, disable=['parser', 'ner'])
    # Do lemmatization
    dataLemmatized = lemmatization(dataWordsNgrams)
       
    return dataLemmatized
    

In [8]:
# Main text analysis pipeline
# This cell performs the complete text analysis workflow:
# 1. Reads and tokenizes a text file from the data directory
# 2. Removes punctuation and stop words
# 3. Builds bigram and trigram models
# 4. Lemmatizes tokens using spaCy
# 5. Computes word frequencies
# 6. Generates an interactive histogram of the top N words
# TODO Naming:
n = 10
documentName = 'Hamlet.txt' # TODO Inputs
outputFile = "topTenPlainText" # TODO Inputs
fmt = '.html'
figure_Xlabel = "Word"
figure_Ylabels = "Count"
figureZlabel = "Percent"
figureWidth = 750
figure_height = 550
figure_Axisangle = -45
figureTitle = 'Top 10 Words, Hamlet'
colors = px.colors.qualitative.Dark24
labCol = "crimson"

textFilepath = os.path.join(data_path, documentName) # TODO Inputs
# Read and tokenize the input text file line by line
docs=[]
with open(textFilepath, "r", encoding = encoding, errors = how_to_handleErrors) as f:
    for line in f:
        singleLine = line.strip()
        if len(singleLine) == 0:
            continue
        docs.append(singleLine.split())
# remove punctuation from sentences and parse into words
words = []
for sentence in docs:
    words.append(simple_preprocess(str(sentence), deacc=True)) # deacc=True removes punctuations
# remove stop words
stop=[]
for doc in words:
    processed_doc = []
    for word in simple_preprocess(str(doc)):
        if word not in stopWords:
            processed_doc.append(word)
    stop.append(processed_doc)
lemma = makeLemma(stop)
tokens=[]
for sublist in  lemma:
    for item in sublist:
        tokens.append(item)
# get frequency
freq = defaultdict(int)

for t in tokens:
    freq[t] += 1
# sort frequency in descending order
freq = sorted(freq.items(), key = operator.itemgetter(1), reverse = True)
imgFilepath = os.path.join(resultsPath, outputFile + fmt)

# plot top ten words in a histogram
topn = n
df = pd.DataFrame(freq, columns = ["Words", "Count"])
df["Pct"] = ((df["Count"]/df["Count"].sum())*100).round(3)
df["Pct"] = df["Pct"].astype(str) + "%"
# TODO DRY:
dfPct = df[0:1]
dfPct = pd.concat([dfPct,df[1:2]])
dfPct = pd.concat([dfPct,df[2:3]])
dfPct = pd.concat([dfPct,df[3:4]])
dfPct = pd.concat([dfPct,df[4:5]])
dfPct = pd.concat([dfPct,df[5:6]])
dfPct = pd.concat([dfPct,df[6:7]])
dfPct = pd.concat([dfPct,df[7:8]])
dfPct = pd.concat([dfPct,df[8:9]])
dfPct = pd.concat([dfPct,df[9:10]])
# Set y-axis range for the histogram plot
high = max(df["Count"])
low = 0
fig = px.bar(dfPct, x = "Words", y = "Count",hover_data=[dfPct["Pct"]],text = "Count", color = "Words", 
             title = figureTitle, color_discrete_sequence=colors,
             labels = {"Words":figure_Xlabel,"Count":figure_Ylabels,"Pct":figureZlabel})
fig.update_layout(title={'y':0.90, 'x':0.5, 'xanchor': 'center', 'yanchor':'top'}, 
                  font={"color": labCol}, width = figureWidth, height = figure_height, showlegend=False)
fig.update_xaxes(tickangle = figure_Axisangle)
fig.update_yaxes(range = [low,math.ceil(high + 0.1 * (high - low))])
py.offline.plot(fig, filename=imgFilepath, auto_open = False)
fig.show()
